# 03 — Entrenamiento del modelo de riesgo

Entrena un clasificador binario que estima la probabilidad de diagnóstico de cáncer de mama a partir de los 10 factores BCSC. Guarda el modelo en `backend/models/modelo_riesgo.pkl` para que la API lo cargue en producción.

**Importante:** los datos están agregados con la columna `count`. Expandimos por `count` (o usamos `sample_weight`) para que el modelo represente correctamente la población.

In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, roc_auc_score,
                             confusion_matrix, RocCurveDisplay)
import matplotlib.pyplot as plt

df = pd.read_parquet('../data/clean/bcsc_clean.parquet')
FEATURES = ['age_group_5_years', 'race_eth', 'first_degree_hx',
            'age_menarche', 'age_first_birth', 'BIRADS_breast_density',
            'current_hrt', 'menopaus', 'bmi_group', 'biophx']
TARGET = 'breast_cancer_history'

X = df[FEATURES].values
y = df[TARGET].values
w = df['count'].values  # sample_weight respeta la agregación
print('Filas:', len(df), '· Mamografías representadas:', int(w.sum()))

In [ ]:
X_tr, X_te, y_tr, y_te, w_tr, w_te = train_test_split(
    X, y, w, test_size=0.2, random_state=42, stratify=y
)
print('Train:', X_tr.shape, '· Test:', X_te.shape)

In [ ]:
# Baseline: Regresión Logística
logreg = LogisticRegression(max_iter=2000, class_weight='balanced')
logreg.fit(X_tr, y_tr, sample_weight=w_tr)
p_lr = logreg.predict_proba(X_te)[:, 1]
print('AUC LogReg:', roc_auc_score(y_te, p_lr, sample_weight=w_te))

In [ ]:
# Random Forest
rf = RandomForestClassifier(
    n_estimators=200, max_depth=12, n_jobs=-1,
    class_weight='balanced', random_state=42,
)
rf.fit(X_tr, y_tr, sample_weight=w_tr)
p_rf = rf.predict_proba(X_te)[:, 1]
print('AUC RandomForest:', roc_auc_score(y_te, p_rf, sample_weight=w_te))

imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
imp.plot.barh(title='Importancia de variables (RF)'); plt.gca().invert_yaxis(); plt.show()

In [ ]:
# Curva ROC comparativa
fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_predictions(y_te, p_lr, name='LogReg', ax=ax, sample_weight=w_te)
RocCurveDisplay.from_predictions(y_te, p_rf, name='RandomForest', ax=ax, sample_weight=w_te)
plt.show()

In [ ]:
# Elegimos el mejor (típicamente RF) y lo guardamos
modelo_final = rf
out = Path('../backend/models/modelo_riesgo.pkl')
out.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(modelo_final, out)
print(f'Modelo guardado en {out.resolve()}')

## Mapeo de probabilidad a niveles de riesgo

El backend (`backend/app/predictor.py`) usa estos cortes:

- `prob < 0.10` → **Bajo**
- `0.10 ≤ prob < 0.25` → **Medio**
- `prob ≥ 0.25` → **Alto**

Estos cortes se pueden recalibrar mirando los percentiles de la distribución de probabilidades en el set de prueba para que cada nivel agrupe ~33% de la población, o ajustarlos según la sensibilidad clínica deseada.